In [ ]:
"""TTS generator notebook for creating MP3 output."""

from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

DEFAULT_TTS_MODELS = [
    "gpt-4o-mini-tts",
    "tts-1",
    "tts-1-hd",
]


class TTSGenerator:
    def __init__(self):
        self.client = OpenAI()

    def text_to_audio(self, text: str, output_path="output.mp3", model: str | None = None):
        speech_file_path = Path(output_path)

        candidate_models = []
        if model:
            candidate_models.append(model)
        candidate_models.extend(m for m in DEFAULT_TTS_MODELS if m not in candidate_models)

        last_error = None
        for candidate_model in candidate_models:
            try:
                with self.client.audio.speech.with_streaming_response.create(
                    model=candidate_model,
                    voice="alloy",
                    input=text,
                ) as response:
                    response.stream_to_file(speech_file_path)
                return str(speech_file_path)
            except Exception as exc:
                last_error = exc
                continue

        raise RuntimeError(
            "Unable to generate speech with any supported TTS model. "
            "Tried: " + ", ".join(candidate_models)
        ) from last_error
